In [ ]:
# /greenball.mp4

import cv2
import numpy as np
from google.colab.patches import cv2_imshow  # 콜랩에서 이미지 출력 지원

video_path = '/greenball.mp4'

In [ ]:
def track(image):
  # BGR 이미지(numpy array) 입력 >> 초록색 물체 중심좌표(centroid) 추적

  # 1) 노이즈 제거 >> 블러(blur)처리
  blur = cv2.GaussianBlur(image, (5,5),0)
  # (5,5) : 5*5 filter
  # 0 : 표준편차 자동 계산

  # 2) BGR >> HSV 색 공간 변환
  hsv = cv2.cvtColor(blur, cv2.COLOR_BGR2HSV)
  # HSV (Hue, Saturation, Value)
  # hue : 색상(빨강, 초록, 파랑...) / Saturation: 채도(색이 진하냐) / Value(명도, 빛의 밝기)
  # HSV : 조명이 바뀌어도 색상 찾기가 더 쉬움
  # BGR : 컴퓨터 중심(색 검출이 힘들어요 파랑,그린,빨강 위주, 조명의 밝기 영향을 많이 받아요)

  # 3) 초록색 범위 설정(필요하면 값 조정 가능)
  lower_green = np.array([40, 70, 70]) # [색상 40, 채도 70, 명도 70]
  upper_green = np.array([80, 200, 200])
  # 연한 녹색 - 진한 녹색 범위 설정

  # 4) 마스크 생성
  mask = cv2.inRange(hsv, lower_green, upper_green)
  # 초록색 부분만 하얀색(255), 나머지는 검은색(0) 만들기
  # 초록색 부분 오려내는 효과

  # lower_green - pixel value - upper_green
  # 범위 안에 있으면 >>> 255 (흰색) / 없으면 >>> 0(검정색)
  # >> 결과물을 mask 에 저장 (mask 이미지 흑백(binary : 초록색>>흰색 1, 아니면 검정색 0))

  # 5) 마스크를 다시 블러링 (잡음 제거)
  bmask = cv2.GaussianBlur(mask, (5,5),0)
  # binary mask (bmask) (이미지가 0(code 0)또는 255(code 1) 구성된 영상-흑백 영상)

  # 6) 모멘트(moment) 이용하여 중심 좌표 계산
  moments = cv2.moments(bmask)
  # key-value {}로 저장 {m00: ...면적(area)}
  # m00: 전체 면적(하얀 픽셀 개수)
  m00 = moments['m00']

  # 중심좌표(centroid) 초기화
  centroid_x, centroid_y = None, None

  # 만약 면적이 0이 아니라면(면적이 존재한다면)
  if m00 !=0:
    centroid_x = int(moments['m10']/m00) # 한 영역의 중심 x 좌표
    centroid_y = int(moments['m01']/m00) # 한 영역의 중심 y 좌표

  # 기본 값: 중심 없음
  # >> 초록색을 못 찾으면
  # 왜 -1 을 썼는가? 유효하지 않은 좌표 (x좌표, y좌표 >=0 >> 이미지 화면에 보이는 것은 (0,0)~(w,h))
  # center point 없어 >> 객체 못 찾았어 >> 객체가 있다면 (중심점이 있다면) >> (cx, cy)
  crt = (-1, -1)

  # 중심이 계산된 경우
  if centroid_x != None and centroid_y != None:
    ctr = (centroid_x, centroid_y)
    # 이미지 위에 중심점 표시(검은색 점)
    cv2.circle(image, ctr, 4, (0,0,0),-1)
    # 4 : radius(반지름), -1: 원의 내부를 지정된 색으로 채워라

  cv2_imshow(image) # colab 에서 cv2.imshow() 대신 사용

  return ctr


# 2. 메인 실행 부분(동영상 캡처)

if __name__ == '__main__':
  # 프로그램 시작점(이 파일을 직접 실행할 때만 동작)
    capture = cv2.VideoCapture(video_path)

    if not capture.isOpened():
        print('동영상을 열 수 없습니다. 경로를 확인하세요.')
    else:
      frame_idx = 0

      while True:
        okay, image = capture.read()

        # 비디오가 마지막이라면
        if not okay:
          print('영상 끝까지 재생 완료')
          break

        # 초록색 공 추적
        ctr = track(image)
        print(f'Frame {frame_idx}: centroid = {ctr}')

        frame_idx += 1

      capture.release()
      # cv2.destroyAllWindows()

왜 HSV 로 변환해야 하나?

- 이유 1. 색 기반 객체 추출이 쉬워짐

BGR은 파란색, 초록색, 빨간색의 밝기 조합이기 때문에
예를 들어 노란색 객체를 찾고 싶을 때, BGR로는 값 범위를 지정하기 어렵습니다.

하지만 HSV에서는:

- Hue(H): 어떤 색인지 (0~179)

- Saturation(S): 얼마나 진한지

- Value(V): 밝기

예를 들어 노랑색(노란 공, 노란 차선)을 찾는다면:

[예제]

- lower_yellow = np.array([20, 100, 100])
- upper_yellow = np.array([30, 255, 255])


이렇게 한 번에 색 범위 지정이 가능합니다.

이유 2. 조명 변화(밝기)에 강함

실제 영상처리에서는 조명, 그림자, 햇빛, 실내 조명 등 빛의 영향이 매우 큽니다.

- BGR에서는 조명 변화 → 픽셀 값 변화 → 색 분리가 어려움
- HSV에서는 Value(V, 밝기) 값을 별도로 다루기 때문에
조명 변화에도 Hue(색상) 값은 크게 변하지 않음 → 안정적인 색 검출 가능

이유 3. 실시간 영상 처리를 쉽게 함

예:

손 인식(피부색 thresholding), 라인 트래킹 로봇 (노란 차선 인식), 공 색 추적, 객체 추적 (mean-shift, camshift)
주차선 인식, 신호등 빨간/초록 검출 >> 모두 HSV 활용

모멘트 주요 값 정리

- m00 : 한 영역(area) 전체 면적(픽셀 수) 영역 크기 계산
- m10, m01 : 1차 모멘트 (무게 중심 계산)
- m20, m02:  2차 모멘트(회전, 분산, 방향 관련 계산)